## 02 - Build Expression Matrix

### Aim
* Turn the raw `SummarizedExperiment` from notebook 01 into flat, analysis-ready tables: a raw TPM matrix (genes x samples, gene-symbol rows), a raw counts matrix (reference/QC only), and a sample metadata table.

### Input
* `data/raw/TCGA-PAAD_RNAseq_SE.rds` produced by notebook 01.

### Output
* `data/processed/TCGA-PAAD_TPM_raw.tsv` - linear-scale (non-log) TPM matrix, gene-symbol rows x sample-barcode columns.
* `data/processed/TCGA-PAAD_counts_raw.tsv` - raw counts matrix, same gene/sample set (kept for reference/QC only).
* `data/processed/TCGA-PAAD_sample_metadata.tsv` - one row per sample/aliquot, including sample type, tumor grade/stage and tumor-purity annotations (ABSOLUTE purity, pathologist-reviewed cellularity).

### Steps
* Extract the `tpm_unstrand` and `unstranded` assays from the SE.
* Collapse Ensembl-ID rows to gene symbols: for each duplicated symbol, keep the row with the highest mean TPM across samples (the convention most deconvolution signature matrices, e.g. LM22, expect), and drop rows with missing/empty symbols.
* Build the sample metadata table from `colData(se)`.
* Write all three tables to `data/processed/`.

### Result
* Collapsed 60,660 Ensembl rows to 59,427 unique gene symbols (dropped 1,233 duplicate/unnamed rows).
* Final TPM matrix: 59,427 genes x 182 samples. Metadata for 182 samples.

### Conclusion
* An analysis-ready, linear-scale TPM matrix is produced, suitable as direct input to TME deconvolution tools (CIBERSORT, EPIC, quanTIseq, xCell, MCPcounter). Proceed to notebook 03 for QC before deconvolution.


### Set working directory to the project root

In [1]:
# Jupyter's R kernel starts with the notebook's own folder as the working
# directory. This pipeline's root is progress/ (scripts/ and data/ live
# nested under progress/, not the repo root) - every path below is
# relative to that.
if (basename(getwd()) == "rnaseq_download") setwd(file.path("..", ".."))
getwd()

[1] "/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer"

### Load packages and the raw SummarizedExperiment

In [2]:
library(SummarizedExperiment)
library(dplyr)
library(readr)

raw_dir  <- "data/raw"
out_dir  <- "data/processed"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

se <- readRDS(file.path(raw_dir, "TCGA-PAAD_RNAseq_SE.rds"))

Loading required package: MatrixGenerics



Loading required package: matrixStats




Attaching package: ‘MatrixGenerics’




The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges, rowRanks,
    rowSdDiffs, rowSds, rowSums2, rowTabulates, rowVarDiffs, rowVars,
    rowWeightedMads, rowWeightedMeans, rowWeightedMedians,
    rowWeightedSds, rowWeig

Loading required package: GenomicRanges



Loading required package: stats4



... (25 outputs left out)

The following object is masked from ‘package:generics’:

    explain




The following object is masked from ‘package:matrixStats’:

    count




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




### Extract TPM and counts assays
`tpm_unstrand` is linear-scale TPM (NOT log-transformed) - the matrix requested. `unstranded` is raw counts, kept for QC/reference only.

In [3]:
stopifnot("tpm_unstrand" %in% assayNames(se))
tpm_raw    <- assay(se, "tpm_unstrand")
counts_raw <- assay(se, "unstranded")

gene_info <- as.data.frame(rowData(se)) %>%
  select(gene_id, gene_name, gene_type) %>%
  mutate(mean_tpm = rowMeans(tpm_raw, na.rm = TRUE))

### Collapse Ensembl IDs to gene symbols
Drop rows with missing/empty gene symbol, then keep the highest-expressed row per duplicated symbol (avoids silently summing unrelated loci).

In [4]:
keep_idx <- gene_info %>%
  mutate(row_idx = row_number()) %>%
  filter(!is.na(gene_name), gene_name != "") %>%
  group_by(gene_name) %>%
  slice_max(order_by = mean_tpm, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  pull(row_idx)

tpm_mat    <- tpm_raw[keep_idx, , drop = FALSE]
counts_mat <- counts_raw[keep_idx, , drop = FALSE]
rownames(tpm_mat)    <- gene_info$gene_name[keep_idx]
rownames(counts_mat) <- gene_info$gene_name[keep_idx]

cat(sprintf(
  "Collapsed %d Ensembl rows -> %d unique gene symbols (dropped %d duplicate/unnamed rows).\n",
  nrow(tpm_raw), nrow(tpm_mat), nrow(tpm_raw) - nrow(tpm_mat)
))

Collapsed 60660 Ensembl rows -> 59427 unique gene symbols (dropped 1233 duplicate/unnamed rows).


### Build the sample metadata table

In [5]:
sample_meta <- as.data.frame(colData(se)) %>%
  select(
    barcode, patient, sample, sample_type, definition,
    shortLetterCode, preservation_method, tissue_type, days_to_birth,
    tumor_grade, ajcc_pathologic_stage,
    tumor_purity_absolute = `paper_ABSOLUTE.Purity`,
    tumor_purity_pathologist = `paper_Pathologist.Reviewed.Tumor.Cellularity`
  ) %>%
  distinct()

### Write the TPM matrix, counts matrix and sample metadata

In [6]:
write_tsv(
  as.data.frame(tpm_mat) %>% tibble::rownames_to_column("gene_symbol"),
  file.path(out_dir, "TCGA-PAAD_TPM_raw.tsv")
)
write_tsv(
  as.data.frame(counts_mat) %>% tibble::rownames_to_column("gene_symbol"),
  file.path(out_dir, "TCGA-PAAD_counts_raw.tsv")
)
write_tsv(sample_meta, file.path(out_dir, "TCGA-PAAD_sample_metadata.tsv"))

cat(sprintf(
  "Wrote TPM matrix (%d genes x %d samples) and metadata for %d samples to %s\n",
  nrow(tpm_mat), ncol(tpm_mat), nrow(sample_meta), out_dir
))

Wrote TPM matrix (59427 genes x 182 samples) and metadata for 182 samples to data/processed
